# Silver_to_Gold — Camada Gold · Star Schema + Analytics

Constrói as dimensões, a fato e as tabelas-ponte (*bridge*) a partir da Silver.

**Decisões de modelagem:**
- **Surrogate keys** `BIGINT` via `row_number()` ordenado pela chave natural (determinístico e sem colisões, ao contrário de hash truncado). Para o volume do projeto (~100 mil filmes) o custo da janela global é irrelevante.
- **Fato com grão = 1 filme** e somente filmes **Lançados** (conforme enunciado: "consolidar métricas de filmes lançados"). Os joins com financeiro/engajamento são 1:1 (Silver já deduplicada por filme) e há uma **asserção de unicidade** de grão.
- `dim_people` é única por **(nome, tipo)**: a mesma pessoa pode ser Ator e Diretor, e o `tipo_pessoa` fica na dimensão (as pontes só ligam filme↔pessoa).
- **Pontes** M:N (`genre`, `person`, `company`) evitam duplicar linhas da fato.
> **Execução orquestrada:** este notebook é a task `to_Gold` do Databricks Workflow `CineData_Medallion_Pipeline` (depende de `to_Silver`). Todos os parâmetros são *widgets* com valor padrão, então ele roda sem intervenção (interativo ou via Job).

**Estrutura:** (1) dimensões, fato e pontes · (2) validação de integridade · (3) Desafio de Analytics (6 perguntas com `display()`).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalogo", "workspace", "Catálogo (Unity Catalog)")
CATALOGO = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Camada Gold: Star Schema para BI/IA'")


def gravar_gold(df, tabela):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"{tabela:34s} linhas={spark.table(tabela).count():>9,}")


def surrogate_key(ordem):
    """Chave substituta sequencial BIGINT, determinística (ordenada pela chave natural)."""
    return F.row_number().over(Window.orderBy(*ordem)).cast("bigint")

## Dimensões

In [0]:
# ---------------- gold.dim_movies ----------------
dim_movies = (
    spark.table("silver.tb_info_filmes")
    .select(
        surrogate_key([F.col("id_filme")]).alias("sk_movie_id"),
        F.col("id_filme").cast("string").alias("id_filme"),
        "titulo",
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        "idioma_original",
        "status_filme",
        "sinopse",
    )
)
gravar_gold(dim_movies, "gold.dim_movies")

# ---------------- gold.dim_genres ----------------
dim_genres = (
    spark.table("silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct()
    .select(surrogate_key([F.col("nome_genero")]).alias("sk_genre_id"), "nome_genero")
)
gravar_gold(dim_genres, "gold.dim_genres")

# ---------------- gold.dim_people (Ator, Diretor, Roteirista) ----------------
dim_people = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct()
    .select(surrogate_key([F.col("tipo_pessoa"), F.col("nome_pessoa")]).alias("sk_person_id"), "nome_pessoa", "tipo_pessoa")
)
gravar_gold(dim_people, "gold.dim_people")

# ---------------- gold.dim_companies ----------------
dim_companies = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora")).distinct()
    .select(surrogate_key([F.col("nome_produtora")]).alias("sk_company_id"), "nome_produtora")
)
gravar_gold(dim_companies, "gold.dim_companies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_movies                    linhas=   97,611


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_genres                    linhas=       19
gold.dim_people                    linhas=  416,599
gold.dim_companies                 linhas=   45,037


## Tabela Fato — `gold.fact_movies_performance`

In [0]:
fin = spark.table("silver.tb_financeiro_filmes")
met = spark.table("silver.tb_metricas_engajamento")

fact = (
    spark.table("gold.dim_movies")
    .filter(F.col("status_filme") == "Lançado")                  # fato = filmes lançados
    .select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")                                # 1:1 (silver deduplicada)
    .join(met, "id_filme", "left")                                # 1:1 (silver deduplicada)
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

# Garantia do grão: 1 linha por filme (joins não podem duplicar)
total, distintos = fact.count(), fact.select("sk_movie_id").distinct().count()
assert total == distintos, f"Grão violado: {total} linhas para {distintos} filmes"
gravar_gold(fact, "gold.fact_movies_performance")

gold.fact_movies_performance       linhas=   96,261


## `gold.dim_reviews` (métrica resumida por filme)

In [0]:
resumo_reviews = (
    spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),                  # total de avaliações do filme
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),  # média ignora notas NULL
    )
)

dim_reviews = (
    resumo_reviews
    .join(spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"), "id_filme", "inner")  # só filmes do catálogo
    .select(surrogate_key([F.col("sk_movie_id")]).alias("sk_review_id"),
            "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
gravar_gold(dim_reviews, "gold.dim_reviews")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_reviews                   linhas=   27,226


## Tabelas-ponte (M:N) — evitam duplicar a fato

In [0]:
mov = spark.table("gold.dim_movies").select("sk_movie_id", "id_filme")

# ---- bridge_movie_genre ----
bridge_movie_genre = (
    spark.table("silver.tb_generos")
    .join(mov, "id_filme", "inner")
    .join(spark.table("gold.dim_genres"), F.col("genero") == F.col("nome_genero"), "inner")
    .select("sk_movie_id", "sk_genre_id").distinct()
)
gravar_gold(bridge_movie_genre, "gold.bridge_movie_genre")

# ---- bridge_movie_person (chave composta nome + tipo) ----
bridge_movie_person = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(mov, "id_filme", "inner")
    .join(spark.table("gold.dim_people"),
          (F.col("nome_entidade") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")), "inner")
    .select("sk_movie_id", "sk_person_id").distinct()
)
gravar_gold(bridge_movie_person, "gold.bridge_movie_person")

# ---- bridge_movie_company ----
bridge_movie_company = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(mov, "id_filme", "inner")
    .join(spark.table("gold.dim_companies"), F.col("nome_entidade") == F.col("nome_produtora"), "inner")
    .select("sk_movie_id", "sk_company_id").distinct()
)
gravar_gold(bridge_movie_company, "gold.bridge_movie_company")

gold.bridge_movie_genre            linhas=  139,860
gold.bridge_movie_person           linhas=  756,470
gold.bridge_movie_company          linhas=  118,073


## Validação de integridade referencial (todas as contagens de órfãos devem ser 0)

In [0]:
def orfaos(filha, col_fk, pai, col_pk):
    """Conta FKs sem correspondente na dimensão (left_anti). Colunas renomeadas para evitar
    referências ambíguas entre DataFrames (exigência do Spark Connect)."""
    fk = spark.table(filha).select(F.col(col_fk).alias("fk")).distinct()
    pk = spark.table(pai).select(F.col(col_pk).alias("pk"))
    return fk.join(pk, F.col("fk") == F.col("pk"), "left_anti").count()

verificacoes = [
    ("gold.fact_movies_performance", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.dim_reviews", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_genre", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_genre", "sk_genre_id", "gold.dim_genres", "sk_genre_id"),
    ("gold.bridge_movie_person", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_person", "sk_person_id", "gold.dim_people", "sk_person_id"),
    ("gold.bridge_movie_company", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_company", "sk_company_id", "gold.dim_companies", "sk_company_id"),
]
for filha, fk, pai, pk in verificacoes:
    print(f"{filha}.{fk:14s} -> {pai}.{pk:14s} órfãos: {orfaos(filha, fk, pai, pk)}")

gold.fact_movies_performance.sk_movie_id    -> gold.dim_movies.sk_movie_id    órfãos: 0
gold.dim_reviews.sk_movie_id    -> gold.dim_movies.sk_movie_id    órfãos: 0
gold.bridge_movie_genre.sk_movie_id    -> gold.dim_movies.sk_movie_id    órfãos: 0
gold.bridge_movie_genre.sk_genre_id    -> gold.dim_genres.sk_genre_id    órfãos: 0
gold.bridge_movie_person.sk_movie_id    -> gold.dim_movies.sk_movie_id    órfãos: 0
gold.bridge_movie_person.sk_person_id   -> gold.dim_people.sk_person_id   órfãos: 0
gold.bridge_movie_company.sk_movie_id    -> gold.dim_movies.sk_movie_id    órfãos: 0
gold.bridge_movie_company.sk_company_id  -> gold.dim_companies.sk_company_id  órfãos: 0


---
## Desafio de Analytics
Respostas às 6 perguntas de negócio usando **apenas** as tabelas do Star Schema.

> **Data limite (perguntas 5 e 6):** lançamento **mais recente** entre filmes com status *Lançado* e `data_lancamento <= hoje` (ignora datas futuras e não lançados). Recortes: `[limite − 24 meses, limite]` e `[limite − 60 meses, limite]`.

### 1. Receita total (R$) somada de todos os filmes da base

In [0]:
display(spark.sql("""
    SELECT CAST(SUM(receita_brl) AS DECIMAL(24,2)) AS receita_total_brl
    FROM gold.fact_movies_performance
"""))

receita_total_brl
870794835972.79


### 2. Top 5 filmes com maior popularidade

In [0]:
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


### 3. Quantidade de filmes por gênero (do maior para o menor)

In [0]:
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

nome_genero,qtd_filmes
Drama,32127
Documentary,18927
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
TV Movie,4066


### 4. Top 10 filmes por receita (US$ e R$) com `RANK()`

In [0]:
display(spark.sql("""
    SELECT titulo, receita_usd, receita_brl, posicao_ranking
    FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    ) t
    WHERE posicao_ranking <= 10          -- RANK() preserva empates: nenhum filme do top 10 fica de fora
    ORDER BY posicao_ranking, titulo
"""))

titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,15022280000.00,1
Avatar: The Way of Water,2320250281.00,12448374782.59,2
AVENGERS: INFINITY WAR,2052415039.00,11011411925.74,3
spider-man: no way home,1921847111.00,10310901935.23,4
The Lion King,1663075401.00,8922565833.91,5
Top Gun: Maverick,1488732821.00,7987200457.95,6
Barbie,1428545028.00,7664286929.72,7
The Super Mario Bros. Movie,1355725263.00,7273601608.52,8
Black Panther,1349926083.00,7242488427.90,9
Star Wars: The Last Jedi,1332698830.00,7150062492.83,10


### Data limite para as perguntas 5 e 6

In [0]:
limite = spark.sql("""
    SELECT MAX(data_lancamento) AS limite
    FROM gold.dim_movies
    WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
""").first()["limite"]
print("Data limite superior (lançamento mais recente, sem datas futuras):", limite)

Data limite superior (lançamento mais recente, sem datas futuras): 2026-02-19


### 5. Ator com mais participações em filmes lançados nos últimos 2 anos

In [0]:
display(spark.sql(f"""
    SELECT p.nome_pessoa AS ator,
           COUNT(DISTINCT m.sk_movie_id) AS qtd_participacoes
    FROM gold.dim_movies m
    JOIN gold.bridge_movie_person b ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.dim_people p          ON p.sk_person_id = b.sk_person_id
    WHERE p.tipo_pessoa = 'Ator'
      AND m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN add_months(DATE '{limite}', -24) AND DATE '{limite}'
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC, ator
    LIMIT 10
"""))
# A 1ª linha é a resposta; as demais mostram o contexto (e eventuais empates).

ator,qtd_participacoes
Kevin Hart,64
Adlih Torres,12
Andy Dubitsky,12
Cooper Tomlinson,12
Gloria Karel,12
John Simmonds,12
Jonnathon Cripple,12
Sterling L. Pope,12
Tristan Welsh,12
Adriana Asti,6


### 6. Produtora com maior lucro nos últimos 5 anos
Lucro somado por produtora considerando apenas filmes lançados no período e com lucro calculável (receita **e** orçamento conhecidos).

In [0]:
display(spark.sql(f"""
    SELECT c.nome_produtora,
           COUNT(DISTINCT m.sk_movie_id)        AS qtd_filmes_com_lucro,
           SUM(f.lucro_usd)                     AS lucro_total_usd,
           SUM(f.lucro_brl)                     AS lucro_total_brl
    FROM gold.dim_movies m
    JOIN gold.fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
    JOIN gold.bridge_movie_company b    ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.dim_companies c           ON c.sk_company_id = b.sk_company_id
    WHERE m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN add_months(DATE '{limite}', -60) AND DATE '{limite}'
      AND f.lucro_usd IS NOT NULL
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC
    LIMIT 10
"""))
# A 1ª linha é a resposta.

nome_produtora,qtd_filmes_com_lucro,lucro_total_usd,lucro_total_brl
Universal Pictures,24,5772329679.00,30969125960.81
Marvel Studios,8,4953462823.00,26575823391.68
Columbia Pictures,12,3662050755.00,19647268505.66
Pascal Pictures,3,2701952454.00,14496245110.96
Illumination,3,2431353473.00,13044454517.99
Paramount,14,2239394101.00,12014573291.28
Kevin Feige Productions,4,2138205367.00,11471685614.49
20th Century Studios,8,2074815383.00,11131592011.34
Lightstorm Entertainment,1,1860250281.00,9980428782.59
Heyday Films,2,1490495872.00,7996659402.86
